# Endoscapes rubric annotation (frame-level)

In [ ]:
from __future__ import annotations

import json
import random
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, List, Tuple

import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import clear_output, display
from PIL import Image


In [ ]:
# -----------------------------
# CONFIG
# -----------------------------
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'rubrics/cvs_rubrics_v3.json').is_file())
DATASET_ROOT = (REPO_ROOT / "data/endoscapes").resolve()
LABELS_ROOT = REPO_ROOT / "data" / "endoscapes"
RUBRICS_PATH = REPO_ROOT / "rubrics" / "cvs_rubrics_v3.json"
ANNOTATIONS_DIR = REPO_ROOT / "annotations" / "new"

SPLIT = "val"  # train | val | test | val_dev, etc.
SEED = 13
BATCH_IDX = 0
N_PER_LABEL = 10
SHOW_IMAGES = True

print("LABELS_ROOT:", LABELS_ROOT)
print("DATASET_ROOT:", DATASET_ROOT)
print("RUBRICS_PATH:", RUBRICS_PATH)
print("ANNOTATIONS_DIR:", ANNOTATIONS_DIR)


In [ ]:
# -----------------------------
# Helpers (reused from metadata_gt_examples_endoscapes.ipynb)
# -----------------------------

def load_endoscapes_images(split: str, *, labels_root: Path) -> List[Dict[str, object]]:
    ann_path = (labels_root / split / "annotation_ds_coco.json").resolve()
    if not ann_path.exists():
        raise FileNotFoundError(f"Missing annotation file: {ann_path}")
    data = json.loads(ann_path.read_text())
    return list(data.get("images", []))


def criterion_index(criterion: str) -> int:
    idx = {"c1": 0, "c2": 1, "c3": 2}.get(criterion)
    if idx is None:
        raise ValueError(f"Unknown criterion: {criterion}")
    return idx


def resolve_image_path(file_name: str, split: str, *, dataset_root: Path) -> Path:
    return (dataset_root / split / file_name).resolve()


def show_image(path: Path, title: str) -> None:
    if not path.exists():
        print(f"[WARN] Missing image: {path}")
        return
    with Image.open(path) as im:
        img = im.convert("RGB")
    plt.figure(figsize=(4, 3))
    plt.imshow(img)
    plt.title(title)
    plt.axis("off")
    plt.show()


def extract_gt_dict(img: Dict[str, object]) -> Dict[str, float | None]:
    ds = img.get("ds")
    if not isinstance(ds, list) or len(ds) < 3:
        return {"c1": None, "c2": None, "c3": None}
    out: Dict[str, float | None] = {}
    for key, idx in [("c1", 0), ("c2", 1), ("c3", 2)]:
        try:
            out[key] = float(ds[idx])
        except Exception:
            out[key] = None
    return out


def image_key(img: Dict[str, object]) -> str:
    image_id = img.get("id")
    if image_id is not None:
        return f"id:{image_id}"
    return f"file:{img.get('file_name', '')}"


In [ ]:
# -----------------------------
# Load rubrics
# -----------------------------
if not RUBRICS_PATH.exists():
    raise FileNotFoundError(f"Missing rubrics file: {RUBRICS_PATH}")

rubrics_data = json.loads(RUBRICS_PATH.read_text())
criteria_blocks = rubrics_data.get("criteria")
if not isinstance(criteria_blocks, dict):
    raise KeyError("Rubrics missing criteria blocks")
criterion_order = ["C1", "C2", "C3"]

rubric_items: List[Dict[str, object]] = []
rubric_items_by_criterion: Dict[str, List[Dict[str, object]]] = {}
for criterion in criterion_order:
    block = criteria_blocks.get(criterion)
    if not block or "items" not in block:
        raise KeyError(f"Rubrics missing items for {criterion}")
    items = list(block["items"])
    rubric_items_by_criterion[criterion] = items
    for item in items:
        rubric_items.append(
            {
                "criterion": criterion,
                "id": item.get("id"),
                "text": item.get("text"),
                "weight": item.get("weight"),
            }
        )

print("Rubric items:", len(rubric_items))
for criterion in criterion_order:
    print(criterion, "items:", len(rubric_items_by_criterion[criterion]))


In [ ]:
# -----------------------------
# Sample a reproducible batch
# -----------------------------
images = load_endoscapes_images(SPLIT, labels_root=LABELS_ROOT)


def sample_group(
    images: List[Dict[str, object]],
    criterion: str,
    gt_label: int,
    *,
    seed: int,
    batch_idx: int,
    n_per_label: int,
) -> List[Dict[str, object]]:
    idx = criterion_index(criterion)
    group: List[Dict[str, object]] = []
    for img in images:
        ds = img.get("ds")
        if not isinstance(ds, list) or len(ds) <= idx:
            continue
        try:
            gt_val = float(ds[idx])
        except Exception:
            continue
        if gt_val == float(gt_label):
            group.append(img)

    rng = random.Random(seed)
    rng.shuffle(group)

    start = batch_idx * n_per_label
    end = (batch_idx + 1) * n_per_label
    return group[start:end]


group_specs: List[Tuple[str, int]] = [
    ("c1", 1),
    ("c1", 0),
    ("c2", 1),
    ("c2", 0),
    ("c3", 1),
    ("c3", 0),
]

expected_total = len(group_specs) * N_PER_LABEL
all_group_entries: List[Dict[str, object]] = []

for criterion, gt_label in group_specs:
    sampled = sample_group(
        images,
        criterion,
        gt_label,
        seed=SEED,
        batch_idx=BATCH_IDX,
        n_per_label=N_PER_LABEL,
    )
    if len(sampled) < N_PER_LABEL:
        print(
            f"[WARN] {criterion} GT={gt_label} has only {len(sampled)} "
            f"examples for batch {BATCH_IDX}"
        )
    for img in sampled:
        all_group_entries.append(
            {
                "image": img,
                "sample_group": {"criterion": criterion, "gt": int(gt_label)},
            }
        )

# Prefer unique images; allow duplicates only if needed.
final_entries: List[Dict[str, object]] = []
seen = set()
duplicate_entries: List[Dict[str, object]] = []

for entry in all_group_entries:
    key = image_key(entry["image"])
    if key in seen:
        duplicate_entries.append(entry)
        continue
    seen.add(key)
    final_entries.append(entry)

if len(final_entries) < expected_total and duplicate_entries:
    print(
        f"[WARN] {len(duplicate_entries)} duplicates found; "
        "adding duplicates to reach target size"
    )
    for entry in duplicate_entries:
        if len(final_entries) >= expected_total:
            break
        final_entries.append(entry)

if len(final_entries) < expected_total:
    print(
        f"[WARN] Final batch size {len(final_entries)} < target {expected_total}"
    )

print(f"Final batch size: {len(final_entries)} / {expected_total}")


In [ ]:
# -----------------------------
# Annotation UI
# -----------------------------
if not final_entries:
    raise RuntimeError("No images available for annotation.")

ANNOTATIONS_DIR.mkdir(parents=True, exist_ok=True)
run_ts = datetime.now().strftime("%Y%m%d_%H%M%S")
output_path = (
    ANNOTATIONS_DIR
    / f"endoscapes_{SPLIT}__rubrics_v3__seed{SEED}__batch{BATCH_IDX}__{run_ts}.jsonl"
)

CHOICES = ["yes", "no", "uncertain"]

progress_label = widgets.Label()
meta_out = widgets.Output()
image_out = widgets.Output()

item_widgets: Dict[str, widgets.RadioButtons] = {}
item_rows: List[widgets.Widget] = []
header_widgets: Dict[str, widgets.HTML] = {}

for criterion in ["C1", "C2", "C3"]:
    header = widgets.HTML("<h3></h3>")
    header_widgets[criterion] = header
    item_rows.append(header)
    for item in rubric_items_by_criterion[criterion]:
        item_id = item.get("id")
        weight = item.get("weight")
        text = item.get("text")
        label = widgets.HTML(f"<b>{item_id}</b> (w={weight}): {text}")
        rb = widgets.RadioButtons(options=CHOICES, value="uncertain")
        item_widgets[item_id] = rb
        row = widgets.HBox([rb, label])
        item_rows.append(row)

notes_widget = widgets.Textarea(
    value="",
    placeholder="Optional notes...",
    description="Notes:",
    layout=widgets.Layout(width="100%", height="80px"),
)

save_button = widgets.Button(description="Save & Next", button_style="success")
skip_button = widgets.Button(description="Skip", button_style="warning")
reset_button = widgets.Button(description="Reset", button_style="")

status_out = widgets.Output()

state = {"index": 0}


def current_entry() -> Dict[str, object]:
    return final_entries[state["index"]]


def format_gt_label(gt_val: float | None) -> str:
    if gt_val == 1.0:
        return "1"
    if gt_val == 0.0:
        return "0"
    return "unknown"


def render_current() -> None:
    entry = current_entry()
    img = entry["image"]
    sample_group = entry["sample_group"]
    file_name = str(img.get("file_name", ""))
    video_id = img.get("video_id")
    frame_id = img.get("frame_id")
    image_id = img.get("id")
    gt = extract_gt_dict(img)

    progress_label.value = (
        f"Image {state['index'] + 1} / {len(final_entries)}"
    )

    header_widgets["C1"].value = f"<h3>C1 (GT={format_gt_label(gt.get('c1'))})</h3>"
    header_widgets["C2"].value = f"<h3>C2 (GT={format_gt_label(gt.get('c2'))})</h3>"
    header_widgets["C3"].value = f"<h3>C3 (GT={format_gt_label(gt.get('c3'))})</h3>"

    with meta_out:
        clear_output(wait=True)
        print("video_id:", video_id)
        print("frame_id:", frame_id)
        print("file_name:", file_name)
        print("image_id:", image_id)
        print("sample_group:", sample_group)
        print("gt:", gt)

    with image_out:
        clear_output(wait=True)
        if SHOW_IMAGES and file_name:
            img_path = resolve_image_path(file_name, SPLIT, dataset_root=DATASET_ROOT)
            title = f"{video_id} | {frame_id}"
            show_image(img_path, title)


def reset_widgets() -> None:
    for rb in item_widgets.values():
        rb.value = "uncertain"
    notes_widget.value = ""


def build_record(*, skipped: bool) -> Dict[str, object]:
    entry = current_entry()
    img = entry["image"]
    file_name = str(img.get("file_name", ""))
    img_path = resolve_image_path(file_name, SPLIT, dataset_root=DATASET_ROOT)
    gt = extract_gt_dict(img)

    if skipped:
        rubrics = {item_id: "uncertain" for item_id in item_widgets}
    else:
        rubrics = {item_id: rb.value for item_id, rb in item_widgets.items()}

    record = {
        "dataset": "endoscapes",
        "split": SPLIT,
        "seed": SEED,
        "batch_idx": BATCH_IDX,
        "sample_group": entry["sample_group"],
        "image": {
            "file_name": file_name,
            "image_path": str(img_path),
            "video_id": img.get("video_id"),
            "frame_id": img.get("frame_id"),
            "image_id": img.get("id"),
        },
        "gt": gt,
        "rubric_version": "cvs_rubrics_v3",
        "rubrics": rubrics,
        "notes": notes_widget.value.strip(),
        "timestamp": datetime.now(timezone.utc).isoformat(),
    }

    if skipped:
        record["skip_reason"] = "skipped"

    return record


def append_record(record: Dict[str, object]) -> None:
    with output_path.open("a") as f:
        f.write(json.dumps(record) + "\n")


def advance() -> None:
    if state["index"] >= len(final_entries) - 1:
        with status_out:
            clear_output(wait=True)
            print("Done. Reached end of batch.")
        save_button.disabled = True
        skip_button.disabled = True
        return
    state["index"] += 1
    reset_widgets()
    render_current()


def on_save(_):
    record = build_record(skipped=False)
    append_record(record)
    with status_out:
        clear_output(wait=True)
        print("Saved:", output_path.name)
    advance()


def on_skip(_):
    record = build_record(skipped=True)
    append_record(record)
    with status_out:
        clear_output(wait=True)
        print("Skipped:", output_path.name)
    advance()


def on_reset(_):
    reset_widgets()


save_button.on_click(on_save)
skip_button.on_click(on_skip)
reset_button.on_click(on_reset)

render_current()

controls = widgets.HBox([save_button, skip_button, reset_button, progress_label])

ui = widgets.VBox(
    [
        widgets.HTML(f"<b>Output:</b> {output_path}"),
        controls,
        meta_out,
        image_out,
        widgets.VBox(item_rows),
        notes_widget,
        status_out,
    ]
)

display(ui)


In [ ]:
# -----------------------------
# Optional: quick summary of saved annotations
# -----------------------------
if output_path.exists():
    rows = [json.loads(line) for line in output_path.read_text().splitlines() if line.strip()]
    print("Rows:", len(rows))
    if rows:
        item_ids = list(rows[0]["rubrics"].keys())
        uncertain_counts = {item_id: 0 for item_id in item_ids}
        for row in rows:
            for item_id, val in row["rubrics"].items():
                if val == "uncertain":
                    uncertain_counts[item_id] += 1
        top_uncertain = sorted(uncertain_counts.items(), key=lambda x: -x[1])[:5]
        print("Top uncertain items:")
        for item_id, count in top_uncertain:
            print(item_id, count)
